# MD5碰撞攻击

**摘要：** MD5碰撞攻击由王小云教授2004年发现，将MD5碰撞复杂度从2^64降至2^39，彻底颠覆其安全性，本文详解其原理、实现与防护

- **类别：** 现代密码学
- **难度：** 高级
- **预计用时：** 4小时
- **先修：** MD5算法基础、差分密码分析、布尔函数、模运算、哈希函数安全性
- **学习目标：** 掌握MD5碰撞攻击的原理、实现方法及防护策略

> 说明：本课程强调“可运行 + 可解释 + 可练习”。代码尽量仅使用 Python 标准库（Pyodide 友好）。

## 你将获得什么

- **差分分析 核心突破：** 基于差分密码分析的碰撞攻击方法
- **复杂度骤降 2^39：** 碰撞复杂度从2^64降至2^39可实际实现
- **真实碰撞 可验证：** 提供王小云真实碰撞对的验证实现

## 学习路线图（从直觉到实现）

我们把学习过程拆成 4 层，每一层都尽量给出“可验证”的产物：

1. **直觉层**：能说清楚它解决什么安全目标，以及为什么需要“密钥/参数/随机性”。
2. **符号层**：能把关键变换写成一个简短公式，例如 $y=f(x,k)$ 或 $$y=f(x,k)\bmod n$$。
3. **实现层**：能写出可运行的函数/类，并通过至少 3 条 `assert` 自测。
4. **对抗层**：能指出它可能被怎么攻（至少一个思路），并用代码做一个最小实验验证。

> 你会发现：能通过“对抗层”的小实验，往往才算真正理解。


## 应用场景与安全性讨论（扩充阅读）

这一主题通常会在以下场景出现（不同主题侧重点不同）：

- **教学/入门**：用简化模型理解“密钥 + 变换”的思想。
- **工程系统**：用成熟算法与标准协议实现保密性/完整性/认证。
- **攻防分析**：通过已知攻击理解“为什么某些方案不再推荐”。

### 安全目标（Security Goals）

常见目标包括：

- **保密性（Confidentiality）**：未授权者无法获得明文信息。
- **完整性（Integrity）**：消息在传输中被篡改能被发现。
- **认证（Authentication）**：确认对方是谁（或确认数据来自谁）。
- **不可否认（Non-repudiation）**：事后不能否认曾经生成/发送过某信息（通常与签名相关）。

### 威胁模型（Threat Model）

做任何“安全性结论”之前，先明确攻击者能做什么：

- 只能看到密文？还是还能选择明文（chosen-plaintext）？
- 能否篡改、重放、插入消息？
- 能否获取部分密钥/随机数？是否存在侧信道？

> 调试提示：如果你觉得“算法没问题但结果不对”，先检查你实现的威胁模型是否一致——例如你是否在复用 nonce/IV，或者把编码/填充当成了算法的一部分。


## 环境准备

In [ ]:
# 课程通用导入（尽量只用标准库）
import math
import random
import string
import secrets
import hashlib
import hmac
import itertools
import statistics
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Iterable

random.seed(0)  # 为了让演示更可复现


## 热身：数据与表示

很多实现问题来自“数据表示”而不是算法本身。请确保你能区分：

- 文本：`str`
- 字节：`bytes`
- 整数：`int`

并能相互转换。

In [ ]:
msg = "Hello, 密码学"
b = msg.encode("utf-8")
print(type(msg), type(b))  # 预期输出：<class 'str'> <class 'bytes'>
print(b[:10])              # 预期输出：前10个字节（与编码有关）
print(b.hex()[:20])        # 预期输出：十六进制字符串前20个字符


### 工具函数：字节/整数/十六进制

In [ ]:
def bytes_to_hex(b: bytes) -> str:
    return b.hex()

def hex_to_bytes(h: str) -> bytes:
    h = h.strip().lower()
    if h.startswith("0x"):
        h = h[2:]
    return bytes.fromhex(h)

def int_to_bytes(x: int, length: int, byteorder: str = "big") -> bytes:
    return x.to_bytes(length, byteorder=byteorder, signed=False)

def bytes_to_int(b: bytes, byteorder: str = "big") -> int:
    return int.from_bytes(b, byteorder=byteorder, signed=False)

assert hex_to_bytes(bytes_to_hex(b"ABC")) == b"ABC"


## 核心内容分步讲解

### Step 1: 理解MD5碰撞攻击的核心概念

1. MD5碰撞攻击定义：找到两个不同输入产生相同128位MD5摘要的攻击方法，王小云教授2004年的发现从根本上破坏了MD5的安全假设。
2. 技术突破点：将理论碰撞复杂度从2^64（生日攻击）降至实际可实现的2^39，普通计算机几小时内即可完成碰撞生成。
3. 实际危害：可用于数字证书伪造、恶意软件伪装、密码验证绕过等场景，直接威胁基于MD5的安全体系。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 2: 了解MD5碰撞攻击的历史背景

1. 王小云教授的学术贡献：清华大学教授、中科院院士，2004年在CRYPTO会议宣布MD5/SHA-1碰撞攻击，开创了差分密码分析在哈希函数攻击中的系统性应用。
2. MD5的安全演进：1991年发布→1996年发现压缩函数弱点→2004年被完整破解→2006年NIST建议停用→目前仅用于非安全场景。
3. 密码学发展影响：推动了抗碰撞哈希函数研究，加速了全球从MD5向SHA-2/SHA-3/SM3的迁移。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 3: 掌握MD5碰撞攻击的技术原理

1. 差分密码分析核心：分析输入差分（Δx=x1⊕x2）如何通过算法处理产生输出差分（Δy=f(x1)⊕f(x2)），构造高概率的差分传播路径。
2. MD5的结构性弱点：
   - Merkle-Damgård结构易受长度扩展攻击；
   - 压缩函数轮函数非线性度不足，差分易传播；
   - 消息调度设计简单，差分可控性强。
3. 攻击关键步骤：
   - 构造特定差分路径（输入差分模式）；
   - 利用局部碰撞创造中间状态碰撞；
   - 通过消息修改技术控制差分传播；
   - 最终实现完整MD5摘要碰撞。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 4: 构造MD5碰撞攻击的差分路径

1. 差分路径设计：王小云攻击采用的核心差分模式是在特定位置的31位（0x80000000）引入差异，典型模式为：
   - 第一块：位置0、5、10、15各设置0x80000000差分；
   - 第二块：与第一块相同的差分模式，用于抵消第一块的差分影响。
2. 差分特性分析：
   - 汉明重量：仅4个差分位，降低攻击复杂度；
   - 差分概率：约2^(-39)，远高于随机碰撞概率；
   - 位位置选择：精心选择的位置使差分传播概率最大化。
3. 数学基础：通过计算差分路径的概率，优化局部碰撞和差分传播，实现复杂度从2^64到2^39的突破。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 5: 实现MD5碰撞攻击的核心逻辑

1. 差分路径应用：将设计好的差分模式应用到基础消息的特定位置，修改对应32位字的31位值。
2. 碰撞消息生成：
   - 生成128字节（2个512位块）的随机基础消息；
   - 对两个块分别应用差分路径；
   - 构造原始消息和修改消息两对候选碰撞消息。
3. 碰撞验证：计算两对消息的MD5值，验证是否碰撞，分析消息差异是否符合预期的差分模式（0x80）。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 6: 验证王小云的真实MD5碰撞对

1. 真实碰撞对特征：王小云发现的经典碰撞对仅有6处字节差异（均为0x80），但MD5值完全相同（79054025255fb1a26e4bc422aef54eb4）。
2. 差异分析：碰撞对的差异位均出现在精心选择的位置，确保差分通过64轮迭代后相互抵消，最终产生相同的哈希状态。
3. 验证意义：证明MD5碰撞攻击的实际可行性，而非仅停留在理论层面。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 7: 实现完整的MD5碰撞攻击验证程序

1. 核心功能模块：
   - 差分路径构造模块：生成王小云攻击的标准差分模式；
   - 消息修改模块：将差分路径应用到基础消息；
   - 碰撞验证模块：计算MD5值并分析消息差异；
   - 真实碰撞对演示模块：验证王小云的经典碰撞对。
2. 关键实现细节：
   - 小端序处理：MD5采用小端序存储32位字；
   - 差分应用：通过异或操作（^）应用差分模式；
   - 差异分析：逐字节对比消息，验证差分模式正确性。
3. 输出分析：即使随机生成的消息对未直接碰撞，也可验证差分路径的正确应用，理解攻击的核心逻辑。


> 提示：实际MD5碰撞生成需要大量的消息修改和概率性尝试，本实现聚焦于差分路径的构造和验证，而非完整的碰撞搜索

**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
import hashlib
import struct
import random
from typing import List, Tuple, Optional

class MD5DifferentialAttack:
    """MD5差分攻击类 - 实现王小云教授的MD5碰撞攻击核心逻辑"""
    
    def __init__(self):
        # MD5的初始值
        self.initial_values = [0x67452301, 0xEFCDAB89, 0x98BADCFE, 0x10325476]
        
    def construct_differential_path(self) -> Tuple[List[int], List[int]]:
        """
        构造王小云攻击的差分路径
        这是攻击的核心理论部分
        
        返回:
            Tuple[List[int], List[int]]: (第一块差分模式, 第二块差分模式)
        """
        print("=== 构造差分路径 ===")
        
        # 第一块的差分模式 - 王小云攻击的核心差分模式
        # 每个0x80000000表示在第31位（最高位）有差异
        block1_diff = [
            0x80000000, 0x00000000, 0x00000000, 0x00000000,  # 位置0-3
            0x00000000, 0x80000000, 0x00000000, 0x00000000,  # 位置4-7
            0x00000000, 0x00000000, 0x80000000, 0x00000000,  # 位置8-11
            0x00000000, 0x00000000, 0x00000000, 0x80000000,  # 位置12-15
        ]
        
        # 第二块的差分模式 - 用于抵消第一块的差分影响
        block2_diff = [
            0x80000000, 0x00000000, 0x00000000, 0x00000000,  # 位置0-3
            0x00000000, 0x80000000, 0x00000000, 0x00000000,  # 位置4-7
            0x00000000, 0x00000000, 0x80000000, 0x00000000,  # 位置8-11
            0x00000000, 0x00000000, 0x00000000, 0x80000000,  # 位置12-15
        ]
        
        print(f"第一块差分模式: {[hex(d) for d in block1_diff]}")
        print(f"第二块差分模式: {[hex(d) for d in block2_diff]}")
        
        # 分析差分路径的特性
        self._analyze_differential_properties(block1_diff, block2_diff)
        
        return block1_diff, block2_diff
    
    def _analyze_differential_properties(self, diff1: List[int], diff2: List[int]):
        """分析差分路径的数学特性"""
        print("\n=== 差分路径分析 ===")
        
        # 计算汉明重量
        hamming_weight1 = sum(bin(d).count('1') for d in diff1)
        hamming_weight2 = sum(bin(d).count('1') for d in diff2)
        
        print(f"第一块汉明重量: {hamming_weight1}")
        print(f"第二块汉明重量: {hamming_weight2}")
        
        # 分析位位置
        bit_positions1 = []
        bit_positions2 = []
        
        for i, diff in enumerate(diff1):
            if diff != 0:
                bit_pos = 31  # 0x80000000的最高位
                bit_positions1.append((i, bit_pos))
        
        for i, diff in enumerate(diff2):
            if diff != 0:
                bit_pos = 31  # 0x80000000的最高位
                bit_positions2.append((i, bit_pos))
        
        print(f"第一块差异位置: {bit_positions1}")
        print(f"第二块差异位置: {bit_positions2}")
        
        # 计算差分路径概率（王小云攻击的实际概率约为2^(-39)）
        probability = self._calculate_differential_probability(diff1, diff2)
        print(f"差分路径概率: {probability:.2e}")
        print(f"预期尝试次数: {1/probability:.0f}")
    
    def _calculate_differential_probability(self, diff1: List[int], diff2: List[int]) -> float:
        """计算差分路径的概率（简化版）"""
        # 王小云攻击的成功概率约为2^(-39)
        base_prob = 2**(-39)
        
        # 差分位数量调整
        diff_count1 = sum(1 for d in diff1 if d != 0)
        diff_count2 = sum(1 for d in diff2 if d != 0)
        adjustment = 2**(diff_count1 + diff_count2 - 8)
        
        return base_prob * adjustment
    
    def apply_differential_path(self, base_message: bytes, differential_path: List[int]) -> bytes:
        """
        将差分路径应用到基础消息上
        
        Args:
            base_message: 基础消息（必须是512位的倍数）
            differential_path: 差分路径
            
        Returns:
            bytes: 应用差分后的消息
        """
        print(f"\n=== 应用差分路径 ===")
        print(f"原始消息长度: {len(base_message)} 字节")
        
        # 确保消息长度是512位的倍数
        if len(base_message) % 64 != 0:
            raise ValueError("消息长度必须是512位的倍数")
        
        # 将消息转换为32位小端序字数组
        words = []
        for i in range(0, len(base_message), 4):
            word = struct.unpack('<I', base_message[i:i+4])[0]
            words.append(word)
        
        print(f"消息字数量: {len(words)}")
        
        # 应用差分路径
        modified_words = words.copy()
        for i, diff in enumerate(differential_path):
            if i < len(modified_words) and diff != 0:
                original_word = modified_words[i]
                modified_words[i] = original_word ^ diff
                print(f"位置 {i}: 0x{original_word:08x} -> 0x{modified_words[i]:08x}")
        
        # 转换回字节（小端序）
        modified_message = b''
        for word in modified_words:
            modified_message += struct.pack('<I', word)
        
        print(f"修改后消息长度: {len(modified_message)} 字节")
        
        return modified_message
    
    def generate_collision_from_differential(self, differential_path: Tuple[List[int], List[int]]) -> Tuple[bytes, bytes]:
        """
        使用差分路径生成碰撞消息对
        """
        print("\n=== 使用差分路径生成碰撞 ===")
        
        block1_diff, block2_diff = differential_path
        
        # 生成随机的基础消息（128字节 = 1024位 = 2个512位块）
        random.seed(42)  # 固定种子保证可复现
        base_message = bytes([random.randint(0, 255) for _ in range(128)])
        
        print(f"生成的基础消息长度: {len(base_message)} 字节")
        
        # 将消息分成两个512位块
        block1 = base_message[:64]   # 第一个512位块
        block2 = base_message[64:]    # 第二个512位块
        
        print(f"第一块长度: {len(block1)} 字节")
        print(f"第二块长度: {len(block2)} 字节")
        
        # 应用差分路径到第一块
        modified_block1 = self.apply_differential_path(block1, block1_diff)
        
        # 应用差分路径到第二块
        modified_block2 = self.apply_differential_path(block2, block2_diff)
        
        # 构造完整的消息
        message1 = block1 + block2
        message2 = modified_block1 + modified_block2
        
        print(f"\n生成的消息对:")
        print(f"消息1长度: {len(message1)} 字节")
        print(f"消息2长度: {len(message2)} 字节")
        
        return message1, message2
    
    def verify_collision_with_differential(self, msg1: bytes, msg2: bytes, differential_path: Tuple[List[int], List[int]]) -> bool:
        """
        验证碰撞并分析差分路径的效果
        """
        print("\n=== 验证碰撞和差分分析 ===")
        
        # 计算MD5值
        hash1 = hashlib.md5(msg1).hexdigest()
        hash2 = hashlib.md5(msg2).hexdigest()
        
        print(f"消息1 MD5: {hash1}")
        print(f"消息2 MD5: {hash2}")
        print(f"是否碰撞: {hash1 == hash2}")
        
        # 分析消息差异
        self._analyze_message_differences(msg1, msg2)
        
        # 验证差分路径是否正确应用
        self._verify_differential_application(msg1, msg2, differential_path)
        
        return hash1 == hash2
    
    def _analyze_message_differences(self, msg1: bytes, msg2: bytes):
        """分析两个消息的差异"""
        print("\n=== 消息差异分析 ===")
        
        differences = []
        for i, (b1, b2) in enumerate(zip(msg1, msg2)):
            if b1 != b2:
                diff = b1 ^ b2
                differences.append((i, b1, b2, diff))
                print(f"位置 {i:3d}: 0x{b1:02x} -> 0x{b2:02x} (差值: 0x{diff:02x})")
        
        print(f"总差异数: {len(differences)}")
        
        if differences:
            diff_values = [d[3] for d in differences]
            unique_diffs = set(diff_values)
            print(f"唯一差异值: {[hex(d) for d in unique_diffs]}")
            
            # 检查是否符合预期的差分模式
            expected_diff = 0x80  # 0x80000000的最低字节
            if all(d == expected_diff for d in diff_values):
                print("差异模式符合预期 (0x80) ✓")
            else:
                print("差异模式不符合预期 ✗")
    
    def _verify_differential_application(self, msg1: bytes, msg2: bytes, differential_path: Tuple[List[int], List[int]]):
        """验证差分路径是否正确应用"""
        print("\n=== 差分路径验证 ===")
        
        block1_diff, block2_diff = differential_path
        
        # 检查第一块的差分应用
        block1_1 = msg1[:64]
        block1_2 = msg2[:64]
        
        print("第一块差分验证:")
        for i in range(0, len(block1_1), 4):
            word1 = struct.unpack('<I', block1_1[i:i+4])[0]
            word2 = struct.unpack('<I', block1_2[i:i+4])[0]
            word_diff = word1 ^ word2
            expected_diff = block1_diff[i//4]
            
            if word_diff == expected_diff:
                print(f"  位置 {i//4}: 差分正确 (0x{word_diff:08x}) ✓")
            else:
                print(f"  位置 {i//4}: 差分错误 (期望: 0x{expected_diff:08x}, 实际: 0x{word_diff:08x}) ✗")
        
        # 检查第二块的差分应用
        block2_1 = msg1[64:]
        block2_2 = msg2[64:]
        
        print("第二块差分验证:")
        for i in range(0, len(block2_1), 4):
            word1 = struct.unpack('<I', block2_1[i:i+4])[0]
            word2 = struct.unpack('<I', block2_2[i:i+4])[0]
            word_diff = word1 ^ word2
            expected_diff = block2_diff[i//4]
            
            if word_diff == expected_diff:
                print(f"  位置 {i//4}: 差分正确 (0x{word_diff:08x}) ✓")
            else:
                print(f"  位置 {i//4}: 差分错误 (期望: 0x{expected_diff:08x}, 实际: 0x{word_diff:08x}) ✗")
    
    def demonstrate_wang_xiaoyun_collision(self):
        """
        演示王小云发现的真实MD5碰撞对
        这是密码学史上第一个实际的MD5碰撞对
        """
        print("\n=== 王小云真实碰撞对演示 ===")
        
        # 王小云等人发现的第一个MD5碰撞对
        message1_hex = (
            "d131dd02c5e6eec4693d9a0698aff95c2fcab58712467eab4004583eb8fb7f89"
            "55ad340609f4b30283e488832571415a085125e8f7cdc99fd91dbdf280373c5b"
            "d8823e3156348f5bae6dacd436c919c6dd53e2b487da03fd02396306d248cda0"
            "e99f33420f577ee8ce54b67080a80d1ec69821bcb6a8839396f9652b6ff72a70"
        )
        
        message2_hex = (
            "d131dd02c5e6eec4693d9a0698aff95c2fcab50712467eab4004583eb8fb7f89"
            "55ad340609f4b30283e4888325f1415a085125e8f7cdc99fd91dbd7280373c5b"
            "d8823e3156348f5bae6dacd436c919c6dd53e23487da03fd02396306d248cda0"
            "e99f33420f577ee8ce54b67080280d1ec69821bcb6a8839396f965ab6ff72a70"
        )
        
        message1 = bytes.fromhex(message1_hex)
        message2 = bytes.fromhex(message2_hex)
        
        hash1 = hashlib.md5(message1).hexdigest()
        hash2 = hashlib.md5(message2).hexdigest()
        
        print(f"王小云碰撞对验证:")
        print(f"消息1 MD5: {hash1}")
        print(f"消息2 MD5: {hash2}")
        print(f"碰撞验证: {'成功 ✓' if hash1 == hash2 else '失败 ✗'}")
        
        if hash1 == hash2:
            print("这是一个有效的MD5碰撞对！")
            self._analyze_message_differences(message1, message2)
        
        return hash1 == hash2

# ==================== 演示代码 ====================
if __name__ == "__main__":
    # 创建攻击实例
    attack = MD5DifferentialAttack()
    
    # 步骤1: 构造差分路径
    print("\n" + "=" * 40)
    print("步骤1: 构造差分路径")
    print("=" * 40)
    differential_path = attack.construct_differential_path()
    
    # 步骤2: 使用差分路径生成碰撞消息对
    print("\n" + "=" * 40)
    print("步骤2: 使用差分路径生成碰撞消息对")
    print("=" * 40)
    msg1, msg2 = attack.generate_collision_from_differential(differential_path)
    
    # 步骤3: 验证碰撞和差分路径效果
    print("\n" + "=" * 40)
    print("步骤3: 验证碰撞和差分路径效果")
    print("=" * 40)
    is_collision = attack.verify_collision_with_differential(msg1, msg2, differential_path)
    
    # 步骤4: 演示王小云真实碰撞对
    print("\n" + "=" * 40)
    print("步骤4: 王小云真实碰撞对演示")
    print("=" * 40)
    wang_collision = attack.demonstrate_wang_xiaoyun_collision()
    
    # 步骤5: 输出防护建议
    print("\n" + "=" * 40)
    print("步骤5: MD5碰撞攻击防护建议")
    print("=" * 40)
    print("1. 立即停用MD5用于所有安全相关场景")
    print("2. 推荐使用SHA-256/SHA-3/SM3替代MD5")
    print("3. 对关键应用采用算法升级和双重哈希策略")
    print("4. 定期评估密码学算法的安全性")

# 预期输出：根据输入得到对应结果（请与讲解示例对照）

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

### Step 8: MD5碰撞攻击的防护措施

1. 算法替代方案（按推荐优先级）：
   - SHA-256：256位输出，安全且应用广泛，作为最低安全标准；
   - SM3：中国自主设计的256位哈希算法，国密体系核心；
   - SHA-3：最新国际标准，抗量子计算攻击能力强；
   - BLAKE2：高性能安全哈希算法。
2. 最佳实践：
   - 立即停用MD5：在数字签名、证书验证、密码存储等安全场景；
   - 算法升级：逐步将现有MD5应用迁移到安全算法；
   - 双重哈希：过渡期可采用MD5(SHA256(data))的兼容方案；
   - 定期审计：评估系统中使用的哈希算法安全性。
3. 行业标准：遵循NIST、IETF等机构的最新密码学标准，避免使用已被破解的算法。


**深入解释：**  
本步建议你关注两件事：  
1) 这一步在“整体流程图”中处于哪个位置？它是**准备阶段**、**核心变换**还是**验证/输出**？  
2) 它依赖哪些前提（例如参数范围、随机性、互质条件、字节对齐）？  

如果用一个函数表示，本步往往可以写成 $y=f(x,\theta)$，其中 $\theta$ 表示本步的参数集合。  
在实现时，把 $x$、$\theta$ 的类型与范围写清楚，会显著减少 bug。  


**自检：**

- 写出一个最小输入，让本步跑一次。
- 再写一个“故意出错”的输入（例如非法参数），看看报错是否清晰。

In [ ]:
# TODO: 根据本步描述补充最小演示代码
print('TODO')  # 预期输出：TODO

**小实验：随机测试 5 组（模板）**

下面的代码用 `try/except` 包住，避免你还没实现函数时直接报错。你可以逐步把 TODO 替换掉。

In [ ]:
def _maybe_run(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print("（提示）函数尚未准备好或参数不匹配：", type(e).__name__, e)

# TODO: 把 demo_fn 替换为你在本步实现的核心函数名
demo_fn = None

for t in range(5):
    sample = f"case-{t}"
    if callable(demo_fn):
        _maybe_run(demo_fn, sample)
    else:
        print("样例输入：", sample)  # 预期输出：打印 5 行样例输入


**练习：**

1. 给本步写 3 条 `assert` 自测（正常/边界/异常）。
2. 给本步补充一个“可视化”的输出（例如打印中间状态），帮助你定位问题。

> 常见坑：只测“正常输入”很容易漏掉边界问题。

## 综合示例：端到端流程 + 自测

In [ ]:
# 端到端模板：将主题核心操作封装成接口
# 你可以把 encrypt/decrypt 或 hash/sign/verify 等组合在一起

def pipeline(data: bytes) -> bytes:
    # TODO: 替换为你的端到端流程
    return hashlib.sha256(data).digest()

def check_pipeline():
    a = pipeline(b"hello")
    b = pipeline(b"hello")
    c = pipeline(b"hellp")
    assert a == b
    assert a != c
    print("端到端自测通过")  # 预期输出：端到端自测通过

check_pipeline()


## 小实验：敏感性（雪崩效应）

In [ ]:
def hamming_distance_bytes(a: bytes, b: bytes) -> int:
    if len(a) != len(b):
        raise ValueError("长度必须相同才能计算差异度")
    return sum(x != y for x, y in zip(a, b))

def flip_one_bit(data: bytes, bit_index: int = 0) -> bytes:
    if not data:
        return data
    byte_i = bit_index // 8
    bit_i = bit_index % 8
    byte_i = byte_i % len(data)
    mask = 1 << bit_i
    arr = bytearray(data)
    arr[byte_i] ^= mask
    return bytes(arr)

def core_transform(data: bytes) -> bytes:
    # TODO: 替换成你的核心变换
    # 示例：用 SHA-256 作为“对照组”
    return hashlib.sha256(data).digest()

base = b"hello world"
out1 = core_transform(base)
out2 = core_transform(flip_one_bit(base, 0))

print("输出长度(字节):", len(out1))  # 预期输出：32（若使用SHA-256）
print("差异度(字节):", hamming_distance_bytes(out1, out2))  # 预期输出：通常 > 10


## 附录A：最常用的数学与位运算背景

### 1. 模运算（Modular Arithmetic）

当我们写 $$a \equiv b \pmod n$$ 时，意思是 $a$ 与 $b$ 除以 $n$ 的余数相同，也就是 $n$ 整除 $a-b$。

常见等价写法：

- $a \bmod n$：把 $a$ 映射到 $0..n-1$ 的代表元
- 若 $a<0$，Python 仍保证 $a \bmod n \in [0,n-1]$

**一个很重要的直觉：** 模运算会“折叠”整数轴，把无限多的整数映射到有限集合，所以很多密码体制会用到它来构造“封闭”的运算空间。

### 2. 最大公因数与互质

若 $$\gcd(a,n)=1$$，我们称 $a$ 与 $n$ 互质（coprime）。互质非常重要，因为它意味着 $a$ 在模 $n$ 意义下存在乘法逆元 $a^{-1}$，满足：

$$a\cdot a^{-1} \equiv 1 \pmod n$$

### 3. 扩展欧几里得算法（Extended Euclid）

它不仅能算 $\gcd(a,b)$，还能找到整数 $x,y$ 使得：

$$ax+by=\gcd(a,b)$$

当 $\gcd(a,n)=1$ 时，$x$ 就是 $a$ 关于模 $n$ 的逆元（可能为负，需要再取模）。

### 4. 异或（XOR）

在字节层面，异或常写成 $c=a\oplus b$，它有几个极其重要的性质：

- $a\oplus a=0$
- $a\oplus 0=a$
- $(a\oplus b)\oplus b=a$（可逆性）

因此很多流密码/分组模式会用异或把“密钥流”与明文混合。

> 调试提示：如果你发现解密结果不等于明文，先检查“同一份密钥流是否被一致地使用”，以及字节对齐是否正确。


In [ ]:
# 附录A配套代码：gcd、逆元、异或操作的最小实现与自测

def egcd(a: int, b: int) -> Tuple[int, int, int]:
    """返回 (g, x, y) 使得 a*x + b*y = g = gcd(a,b)"""
    if b == 0:
        return (a, 1, 0)
    g, x1, y1 = egcd(b, a % b)
    x = y1
    y = x1 - (a // b) * y1
    return (g, x, y)

def modinv(a: int, n: int) -> int:
    g, x, _ = egcd(a, n)
    if g != 1:
        raise ValueError("不存在逆元：a 与 n 不互质")
    return x % n

print(math.gcd(3, 26))     # 预期输出：1
print(modinv(3, 26))       # 预期输出：9，因为 3*9=27≡1(mod26)

def xor_bytes(a: bytes, b: bytes) -> bytes:
    if len(a) != len(b):
        raise ValueError("xor 需要等长字节串")
    return bytes(x ^ y for x, y in zip(a, b))

p = b"ABC"
k = b"\x01\x01\x01"
c = xor_bytes(p, k)
p2 = xor_bytes(c, k)
print(c)   # 预期输出：b'@BA'（因为 0x41^1=0x40 等）
print(p2)  # 预期输出：b'ABC'


## 常见错误 / 踩坑点 / 调试提示

> 1. **编码问题**：`str`/`bytes` 混用导致哈希/加解密结果不一致。
>
> 2. **参数合法性**：例如需要互质、需要固定长度、需要随机数不可复用。
>
> 3. **端序与位序**：大端/小端混淆、位操作方向写反。
>
> 4. **测试不足**：缺少边界与异常路径测试。
>
> 5. **把演示当安全方案**：课程中的简化实现不等价于生产安全实现。

## 练习题（含更详细要求）

### 练习1：功能完善（必做）
- 目标：把你的核心函数做成“更好用”的版本  
- 要求：
  - 输入支持 `str` 与 `bytes`
  - 对非法参数给出清晰报错（`ValueError`）
  - 至少写 5 条 `assert`（覆盖正常/边界/异常）

### 练习2：最小攻防实验（推荐）
- 目标：实现一个最小的攻击/对抗脚本  
- 示例方向（任选其一）：
  - 暴力枚举 key space
  - 频率分析/统计偏差
  - 重放/篡改检测失败示例
  - 碰撞/相同摘要的构造（仅演示，不鼓励用于真实攻击）

### 练习3：写一段“工程化清单”（可选）
- 目标：把课程知识迁移到真实系统  
- 建议包含：
  - 参数选择（key 长度、随机数、模式）
  - 依赖库与实现来源（优先标准/权威实现）
  - 安全审计点（日志、异常、边界）


In [ ]:
# 练习参考答案框架（示例）
# 说明：这里只提供“结构”，你需要填入你自己的实现。

def solve_ex1(data):
    if isinstance(data, str):
        data_b = data.encode("utf-8")
    elif isinstance(data, (bytes, bytearray)):
        data_b = bytes(data)
    else:
        raise ValueError("只支持 str/bytes")
    return hashlib.sha256(data_b).hexdigest()

assert solve_ex1("a") == solve_ex1("a")
assert solve_ex1("a") != solve_ex1("b")
try:
    solve_ex1(123)
except ValueError:
    pass

print("练习1框架可运行")  # 预期输出：练习1框架可运行


## 小结

把今天的内容压缩成 3 句话：

1. 这个主题的核心变换是 $y=f(x,k)$（或 $$y=f(x,k)\bmod n$$）。
2. 正确实现需要重视：数据表示、参数合法性、测试向量与边界条件。
3. 真正理解来自：能写出最小攻防实验，并解释现象原因。


## 随堂小测（10题）
1. 这个主题的“密钥/参数”是什么？它决定了哪些行为？
2. 为什么需要模运算/置换/异或等操作？分别带来什么效果？
3. 在你的实现里，哪一步最容易写错？你如何用测试发现它？
4. 若攻击者拥有密文（或摘要），最直接的攻击方式是什么？
5. 在什么场景下“能解密”不等于“安全”？举一个例子。
6. 是否存在“重复使用随机数/nonce/IV”的风险？为什么危险？
7. 如果输入非常长，你的实现是否会变慢？瓶颈在哪？
8. 你能写出一个最小“对照实验”，让输出变化更直观吗？
9. 如果要用于真实系统，你会替换/增强哪一部分？
10. 用一句话总结：你学到的最重要概念是什么？

> 自评：能回答 7/10 且能跑通练习，就算达标。
